# 7.6 过程监督与可验证奖励 (Process Supervision)

> 🕐 预估学习时间：40分钟

Outcome Reward（ORM）只看最终答案；Process Reward（PRM）给每一步推理打分。结合可验证奖励（单元测试、数学判定器）是 o1 / DeepSeek-R1 类推理模型的关键训练信号。

本节涵盖：
- ORM vs PRM
- 步骤级标注与自动噪声标签
- 可验证奖励（Verifiable Rewards）
- 与 GRPO / Test-Time Compute 的配合


## 1. ORM vs PRM

| | ORM | PRM |
|-|-----|-----|
| 监督粒度 | 整条轨迹 | 每个推理步 |
| 信用分配 | 难 | 易 |
| 标注成本 | 低 | 高（可用自动判定缓解） |
| 搜索友好 | Best-of-N | 逐步引导 / 树搜索 |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class ORM(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.enc = nn.GRU(d, d, batch_first=True)
        self.head = nn.Linear(d, 1)

    def forward(self, steps):
        # steps: (B, T, d)
        _, h = self.enc(steps)
        return self.head(h[-1]).squeeze(-1)


class PRM(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.enc = nn.GRU(d, d, batch_first=True)
        self.head = nn.Linear(d, 1)

    def forward(self, steps):
        h, _ = self.enc(steps)
        return self.head(h).squeeze(-1)  # (B, T)


d, T, B = 32, 6, 16
# Synthetic trajectories: early mistake should be blamed by PRM
steps = torch.randn(B, T, d)
step_labels = torch.ones(B, T)
step_labels[:, 3:] = 0  # error from step 3 onward
outcome = (step_labels[:, -1] > 0.5).float()

orm = ORM(d)
prm = PRM(d)
opt = torch.optim.Adam(list(orm.parameters()) + list(prm.parameters()), lr=1e-2)

print('=== ORM vs PRM Training ===')
for step in range(80):
    o_pred = orm(steps)
    p_pred = prm(steps)
    loss_o = F.binary_cross_entropy_with_logits(o_pred, outcome)
    loss_p = F.binary_cross_entropy_with_logits(p_pred, step_labels)
    loss = loss_o + loss_p
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 20 == 0 or step == 79:
        print(f'step={step:02d} orm_loss={loss_o.item():.4f} prm_loss={loss_p.item():.4f}')

with torch.no_grad():
    p = torch.sigmoid(prm(steps[:1]))[0]
print(f'\nPRM step probs (expect drop after error): {[f"{x:.2f}" for x in p.tolist()]}')
print(f'Key: PRM localizes credit to the failing step; ORM only knows the trajectory failed.')


## 2. 可验证奖励（Verifiable Rewards）

对数学、代码、形式推理，可用确定性判定器给 0/1 奖励，避免奖励模型黑客：

- 数学：符号等价 / 数值容差
- 代码：单测沙箱
- 工具调用：schema + 执行成功

工业上常 **可验证奖励为主，PRM/ORM 为辅**（覆盖不可自动判定的开放任务）。


In [ ]:
def math_verifier(pred: str, gold: str, tol=1e-6) -> float:
    try:
        p = float(pred.strip())
        g = float(gold.strip())
        return 1.0 if abs(p - g) <= tol else 0.0
    except Exception:
        return 1.0 if pred.strip() == gold.strip() else 0.0


def code_verifier(fn, tests) -> float:
    try:
        for args, expected in tests:
            if fn(*args) != expected:
                return 0.0
        return 1.0
    except Exception:
        return 0.0


def add(a, b):
    return a + b


print('=== Verifiable Rewards ===')
print('math:', math_verifier('42', '42.0'), math_verifier('41', '42'))
print('code:', code_verifier(add, [((1, 2), 3), ((0, 5), 5)]))

# Group advantages for GRPO-style update using verifier rewards
rewards = torch.tensor([1., 0., 1., 0., 0., 1., 0., 1.])
adv = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-8)
print(f'group rewards={rewards.tolist()}')
print(f'group advantages={adv.tolist()}')
print(f'\nKey: Verifier rewards are hard to hack and plug directly into GRPO/RLOO group advantages.')


## 3. 与 Test-Time Compute 的闭环

训练：PRM / 可验证奖励 → GRPO  
推理：Best-of-N / 树搜索 用同一 PRM 打分  

这样训练信号与推理搜索目标一致，避免"训推不一致"。


In [ ]:
def best_of_n(candidates, prm_scores):
    # candidates: list[str], prm_scores: (N, T) step probs -> use min step as trajectory score
    traj_score = prm_scores.min(dim=-1).values
    best = int(traj_score.argmax())
    return best, traj_score


cand_scores = torch.tensor([
    [0.9, 0.9, 0.8, 0.8],
    [0.95, 0.2, 0.9, 0.9],  # early mistake
    [0.85, 0.85, 0.85, 0.84],
])
idx, scores = best_of_n(['A', 'B', 'C'], cand_scores)
print('=== PRM-guided Best-of-N ===')
print(f'trajectory scores={scores.tolist()}')
print(f'selected={idx}')
print(f'Key: Min-step aggregation is conservative and prefers consistently correct chains.')


## 课后思考题

1. 为何仅用 ORM + 长 CoT 容易出现奖励黑客？PRM/验证器如何缓解？
2. 开放域写作任务没有金标时，如何混合 RLAIF 与过程监督？
3. Best-of-N 用 min/ mean/ last-step 聚合 PRM，各有什么偏差？
4. 过程监督数据如何用自动错误注入廉价生成？噪声如何控制？

---
> 本节涵盖了7.6 过程监督与可验证奖励的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
